# Task 1 — Recipe Grid Search (Colab T4)
Chạy thử nghiệm 6 recipe YAML trên 1,000 dòng dữ liệu `xlam_2k` (1 epoch, seed 42) để so sánh 5 Arms với Anchor.

In [ ]:
# --- Cell 1: Gán Token & Clone repo private về Colab GPU ------------------------------
import os, getpass, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem.git"
MARKER = "docs/contracts/cli.md"

# 1. Lấy token từ biến môi trường Colab, nếu chưa có thì hiện ô nhập mật khẩu an toàn
token = os.environ.get("GITHUB_TOKEN")
if not token and not Path(MARKER).exists() and not Path(f"/content/ChatSystem/{MARKER}").exists():
    token = getpass.getpass("Dán GITHUB_TOKEN của bạn vào đây (chữ sẽ bị ẩn): ").strip()
    os.environ["GITHUB_TOKEN"] = token

# 2. Clone repo private về Colab GPU nếu chưa có
if not Path(MARKER).exists():
    if Path(f"/content/ChatSystem/{MARKER}").exists():
        %cd /content/ChatSystem
    elif token:
        print("Đang clone repo private lên ổ đĩa Colab GPU...")
        !git clone --quiet https://{token}@{REPO_HTTPS} /content/ChatSystem
        %cd /content/ChatSystem
    else:
        raise RuntimeError("Thiếu GITHUB_TOKEN để clone repo private!")

REPO = Path.cwd()
print(f"Thành công! Thư mục làm việc trên Colab GPU: {REPO}")

In [18]:
# --- Cell 2: Cài đặt thư viện & Mount Google Drive --------------------------------------
%cd {REPO}

!pip install -q -r requirements-train.txt
!nvidia-smi

# Mount Google Drive theo đúng quy định Mục 5 HUONG_DAN_LAM_VIEC.md
OUT_BASE = "runs"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_BASE = "/content/drive/MyDrive/ChatSystem/runs"
    os.makedirs(OUT_BASE, exist_ok=True)
    print(f"Đã mount Google Drive thành công! Thư mục lưu runs: {OUT_BASE}")
except Exception as e:
    print(f"Lưu ý: Không mount được Drive ({e}). Sử dụng đĩa VM mặc định: {OUT_BASE}")

Mounted at /content/drive
Đã mount Google Drive thành công! Thư mục lưu runs: /content/drive/MyDrive/ChatSystem/runs


In [19]:
# --- Cell 3: Anchor Run (public_anchor.yaml) ----------------------------------------------
%cd {REPO}
out_dir = f"{OUT_BASE}/public_anchor_s42"

!python training/finetune_qlora.py --recipe training/recipes/public_anchor.yaml \
    --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl \
    --output-dir {out_dir}

!python training/predict_toolcall.py --model {out_dir} --data data/public/xlam_2k.eval.json --out results/public/anchor_pred.jsonl
!python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json --pred results/public/anchor_pred.jsonl --out results/public/anchor_results.json

/content/ChatSystem
recipe: training/recipes/public_anchor.yaml → defaults {"model": "Qwen/Qwen2.5-0.5B-Instruct", "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05, "lr": 0.0001, "completion_only": 1, "mask_fn_names": 0.0, "epochs": 1.0, "max_rows": 1000, "seed": 42, "batch_size": 4, "grad_accum": 4, "max_len": 2560, "save_steps": 50, "logging_steps": 10}
GPU: Tesla T4 · compute capability (7, 5) · FORCE_FP16=0 · USE_BF16=False → float16
config.json: 100% 659/659 [00:00<00:00, 3.00MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 91.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 105MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 152MB/s]
Train rows: 1000 · Val rows: 200
[data] rows=1000 max_len=2560 tokens(prompt+completion): max=2426 p95=990 over_limit=0
[val] rows=200 max_len=2560 tokens(prompt+completion): max=1276 p95=937 over_limit=0

model.safetensors: downloading bytes:   5% 47.3M/988M [00:00<00:11, 79

In [21]:
# --- Cell 4: Run 5 Arms -------------------------------------------------------------------
%cd {REPO}
arms = ["recipe_arm_r8", "recipe_arm_lr2e4", "recipe_arm_comp0", "recipe_arm_1b5", "recipe_arm_free"]

for arm in arms:
    print(f"=== Running {arm} ===")
    out_dir = f"{OUT_BASE}/{arm}_s42"
    !python training/finetune_qlora.py --recipe training/recipes/{arm}.yaml \
        --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl \
        --output-dir {out_dir}
    !python training/predict_toolcall.py --model {out_dir} --data data/public/xlam_2k.eval.json --out results/public/{arm}_pred.jsonl
    !python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json --pred results/public/{arm}_pred.jsonl --out results/public/{arm}_results.json

/content/ChatSystem
=== Running recipe_arm_r8 ===
ERROR: --recipe file not found: training/recipes/recipe_arm_r8.yaml
usage: predict_toolcall.py [-h] --eval EVAL --out OUT
                           --backend {openai,transformers}
                           [--base-url BASE_URL] [--model MODEL]
                           [--api-key-env API_KEY_ENV] [--timeout TIMEOUT]
                           [--model-path MODEL_PATH] [--adapter ADAPTER]
                           [--batch-size BATCH_SIZE]
                           [--max-new-tokens MAX_NEW_TOKENS] [--tools TOOLS]
                           [--no-policy] [--system SYSTEM]
                           [--temperature TEMPERATURE] [--limit LIMIT]
predict_toolcall.py: error: the following arguments are required: --eval, --backend
error: --pred file not found: results/public/recipe_arm_r8_pred.jsonl
=== Running recipe_arm_lr2e4 ===
ERROR: --recipe file not found: training/recipes/recipe_arm_lr2e4.yaml
usage: predict_toolcall.py [-h] --eval

In [ ]:
# --- Cell 5: Bootstrap CI & McNemar Test --------------------------------------------------
%cd {REPO}
for arm in arms:
    print(f"=== Comparing Anchor vs {arm} ===")
    !python training/bootstrap_ci.py --run anchor=results/public/anchor_results.json --run {arm}=results/public/{arm}_results.json --pair anchor {arm} --md